In [36]:
import os
import re
import glob
import pickle
import warnings
import json
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Configuration & Constants
# ---------------------------------------------------------------------------
ROLLING_WINDOW = 100
DROP_VAL_BASELINE = True

METRICS = {
    'target_error': {'label': 'Target error [Hz]', 'better': 'lower', 'yscale': 'log', 'ylim': (3, 10), 'bar_ylim': (3, 7)},
    'classification_accuracy': {'label': 'Classification accuracy', 'better': 'higher', 'yscale': 'linear', 'ylim': (0.05, 0.80), 'bar_ylim': (0.4, 0.8)},
}

_LEAF_RE = re.compile(r'^(\d+)_(\d+)$')
_SPLITS = {
    'train': ('trainlog.pkl', ''),
    'val':   ('validationlog.pkl', 'val_'),
    'test':  ('testlog.pkl', 'test_'),
}

# ---------------------------------------------------------------------------
# Self-contained helper functions (from summarize_runs.py)
# ---------------------------------------------------------------------------
def calculate_rolling_mean(data: np.ndarray, rolling_window: int = 1):
    data = np.asarray(data, dtype=float)
    rolling_mean = np.zeros_like(data)
    rolling_std = np.zeros_like(data)
    for i in range(len(data)):
        i1 = max(0, i - rolling_window)
        rolling_mean[i] = data[i1:i + 1].mean()
        rolling_std[i] = data[i1:i + 1].std()
    return rolling_mean, rolling_std

def _load_pickle(path):
    if not os.path.exists(path):
        return None
    with open(path, 'rb') as f:
        return pickle.load(f)

def _identify_run(path):
    leaf = os.path.basename(os.path.normpath(path))
    m = _LEAF_RE.match(leaf)
    if m:
        return int(m.group(1)), int(m.group(2))
    par = _load_pickle(os.path.join(path, 'params.pkl'))
    if par is not None:
        chip = getattr(par, 'chip_id_hidden', getattr(par, 'chip_id_out', None))
        seed = getattr(par, 'weight_seed', None)
        if chip is not None and seed is not None:
            return int(chip), int(seed)
    raise ValueError(f"Could not determine (chip, seed) for run '{path}'.")

def _memmap_metric(path, prefix, name):
    dat_path = os.path.join(path, f"{prefix}{name}.dat")
    if not os.path.exists(dat_path):
        raise FileNotFoundError(f"Missing metric file: {dat_path}")
    shape = None
    for pkl_name, pkl_prefix in _SPLITS.values():
        if pkl_prefix != prefix:
            continue
        meta = _load_pickle(os.path.join(path, pkl_name))
        if isinstance(meta, dict) and name in meta.get('arrays', {}):
            shape = tuple(meta['arrays'][name])
        break
    if shape is None:
        flat = np.memmap(dat_path, dtype='float64', mode='r')
        return np.asarray(flat).reshape(-1, 1)
    return np.memmap(dat_path, dtype='float64', mode='r', shape=shape)

def _accuracy_from_loss(classification_loss, axis):
    return 1.0 - np.asarray(classification_loss).mean(axis=axis)

def load_run(path, skipped_initial_val=False):
    chip, seed = _identify_run(path)
    par = _load_pickle(os.path.join(path, 'params.pkl'))
    batch_size = getattr(par, 'batch_size', 20) if par is not None else 20
    val_frequency = getattr(par, 'val_frequency', None) if par is not None else None
    run = {'chip': chip, 'seed': seed, 'path': path}

    tr_te = np.asarray(_memmap_metric(path, '', 'target_error'))
    tr_cl = np.asarray(_memmap_metric(path, '', 'classification_loss'))
    n_batches = tr_te.shape[0]
    run['train'] = {
        'x': (np.arange(1, n_batches + 1) * batch_size).astype(float),
        'target_error': tr_te.mean(axis=1),
        'classification_accuracy': _accuracy_from_loss(tr_cl, axis=1),
    }

    va_te = np.asarray(_memmap_metric(path, 'val_', 'target_error'))
    va_cl = np.asarray(_memmap_metric(path, 'val_', 'classification_loss'))
    if skipped_initial_val:
        va_te = va_te[:-1]
        va_cl = va_cl[:-1]
        start = 1
    else:
        start = 1 if DROP_VAL_BASELINE else 0
        va_te = va_te[start:]
        va_cl = va_cl[start:]
    n_val = va_te.shape[0]
    if val_frequency is None:
        val_frequency = max(1, n_batches // (n_val + start))
    val_idx = np.arange(start, start + n_val)
    run['val'] = {
        'x': (val_idx * val_frequency * batch_size).astype(float),
        'target_error': va_te.mean(axis=1),
        'classification_accuracy': _accuracy_from_loss(va_cl, axis=1),
    }
    
    te_te = np.asarray(_memmap_metric(path, 'test_', 'target_error'))
    te_cl = np.asarray(_memmap_metric(path, 'test_', 'classification_loss'))
    run['test'] = {
        'target_error': float(te_te.mean()),
        'classification_accuracy': float(1.0 - te_cl.mean()),
    }
    run['best_val'] = {
        'target_error': float(run['val']['target_error'].min()) if n_val else float('nan'),
        'classification_accuracy': float(run['val']['classification_accuracy'].max()) if n_val else float('nan'),
    }
    return run

def load_runs(paths, skipped_initial_val=False):
    runs = []
    for p in paths:
        try:
            runs.append(load_run(p, skipped_initial_val=skipped_initial_val))
        except (FileNotFoundError, ValueError) as e:
            warnings.warn(f"Skipping run '{p}': {e}")
    return runs

def _stack_curves(runs, split, metric, rolling_window):
    if not runs: return None, None
    curves = []
    for r in runs:
        c = np.asarray(r[split][metric], dtype=float)
        if split == 'train' and rolling_window and rolling_window > 1:
            c, _ = calculate_rolling_mean(c, rolling_window)
        curves.append(c)
    lengths = [len(c) for c in curves]
    L = min(lengths)
    stacked = np.stack([c[:L] for c in curves])
    x = np.asarray(runs[0][split]['x'], dtype=float)[:L]
    return x, stacked

def aggregate_group(runs, rolling_window=ROLLING_WINDOW):
    agg = {}
    for metric in METRICS:
        agg[metric] = {}
        for split in ('train', 'val'):
            x, stacked = _stack_curves(runs, split, metric, rolling_window)
            if stacked is None:
                agg[metric][split] = None
            else:
                agg[metric][split] = {
                    'x': x,
                    'mean': stacked.mean(axis=0),
                    'std': stacked.std(axis=0),
                    'n': stacked.shape[0],
                }
    return agg

def _scalar_stats(values):
    a = np.asarray(values, dtype=float)
    a = a[~np.isnan(a)]
    if a.size == 0:
        return {'mean': float('nan'), 'std': float('nan'), 'min': float('nan'), 'max': float('nan'), 'n': 0}
    return {'mean': float(a.mean()), 'std': float(a.std()), 'min': float(a.min()), 'max': float(a.max()), 'n': int(a.size)}

def compute_stats(runs):
    stats = {}
    for metric in METRICS:
        stats[metric] = {}
        for eval_point in ('test', 'best_val'):
            stats[metric][eval_point] = _scalar_stats([r[eval_point][metric] for r in runs])
    return stats

def export_for_typst(full_agg, frozen_agg, filepath="data.json"):
    def extract_curves(agg):
        if agg is None:
            return {"train": [], "val": []}
        acc = agg['classification_accuracy']
        return {
            "train": [
                [float(x), float(m), float(s)] 
                for x, m, s in zip(acc['train']['x'], acc['train']['mean'], acc['train']['std'])
            ],
            "val": [
                [float(x), float(m), float(s)] 
                for x, m, s in zip(acc['val']['x'], acc['val']['mean'], acc['val']['std'])
            ]
        }
    os.makedirs(os.path.dirname(filepath) or '.', exist_ok=True)
    with open(filepath, 'w') as f:
        json.dump({"full": extract_curves(full_agg), "frozen": extract_curves(frozen_agg)}, f)



In [37]:
# Figure 2b: MNIST1D Dynapse Training Curves
mnist1d_frozen_paths = glob.glob('../dynapse_results/mnist1d/frozen/*')
mnist1d_full_paths = glob.glob('../dynapse_results/mnist1d/full/*')

mnist1d_frozen_runs = load_runs(mnist1d_frozen_paths, skipped_initial_val=True)
mnist1d_full_runs = load_runs(mnist1d_full_paths)

mnist1d_frozen_agg = aggregate_group(mnist1d_frozen_runs, ROLLING_WINDOW) if mnist1d_frozen_runs else None
mnist1d_full_agg = aggregate_group(mnist1d_full_runs, ROLLING_WINDOW) if mnist1d_full_runs else None

if mnist1d_frozen_agg and mnist1d_full_agg:
    export_for_typst(mnist1d_full_agg, mnist1d_frozen_agg, filepath="summary_plots/fig2b_dynapse.json")
    print("Exported fig2b_dynapse.json successfully.")



Exported fig2b_dynapse.json successfully.


In [38]:
# Figure 3b: YinYang Dynapse Training Curves
yinyang_frozen_paths = glob.glob('../dynapse_results/yinyang/frozen/*')
yinyang_full_paths = glob.glob('../dynapse_results/yinyang/full/*')

yinyang_frozen_runs = load_runs(yinyang_frozen_paths, skipped_initial_val=True)
yinyang_full_runs = load_runs(yinyang_full_paths)

yinyang_frozen_agg = aggregate_group(yinyang_frozen_runs, ROLLING_WINDOW) if yinyang_frozen_runs else None
yinyang_full_agg = aggregate_group(yinyang_full_runs, ROLLING_WINDOW) if yinyang_full_runs else None

if yinyang_frozen_agg and yinyang_full_agg:
    export_for_typst(yinyang_full_agg, yinyang_frozen_agg, filepath="summary_plots/fig3b_dynapse.json")
    print("Exported fig3b_dynapse.json successfully.")



Exported fig3b_dynapse.json successfully.


In [39]:
# Define helper for Bar Plot exports
def export_bar_plot_data(sim_df, dynapse_frozen_runs, dynapse_full_runs, filepath):
    group_order = ['frozen-hidden', 'locally-learnt', 'fixed-feedback', 'surrogate gradient', 'backprop']
    
    # 1. Process Simulation Data
    sim_df = sim_df[sim_df['note'].isin(group_order)].copy()
    metrics = ['test accuracy (Max)', 'test target error']
    grouped_mean = sim_df.groupby('note')[metrics].mean().reindex(group_order)
    grouped_std = sim_df.groupby('note')[metrics].std().reindex(group_order)
    
    # Create structure
    out = {"simulation": {}, "dynapse": {}}
    for note in group_order:
        acc_m = float(grouped_mean.loc[note, 'test accuracy (Max)']) if not np.isnan(grouped_mean.loc[note, 'test accuracy (Max)']) else 0.0
        acc_s = float(grouped_std.loc[note, 'test accuracy (Max)']) if not np.isnan(grouped_std.loc[note, 'test accuracy (Max)']) else 0.0
        err_m = float(grouped_mean.loc[note, 'test target error']) if not np.isnan(grouped_mean.loc[note, 'test target error']) else 0.0
        err_s = float(grouped_std.loc[note, 'test target error']) if not np.isnan(grouped_std.loc[note, 'test target error']) else 0.0
        out['simulation'][note] = {
            "accuracy_mean": acc_m, "accuracy_std": acc_s,
            "error_mean": err_m, "error_std": err_s
        }
        
    # 2. Process Dynapse Data
    eval_pt = 'best_val'
    dynapse_frozen_stats = compute_stats(dynapse_frozen_runs) if dynapse_frozen_runs else None
    dynapse_full_stats = compute_stats(dynapse_full_runs) if dynapse_full_runs else None
    
    for note in group_order:
        stats = None
        if note == 'frozen-hidden':
            stats = dynapse_frozen_stats
        elif note == 'locally-learnt':
            stats = dynapse_full_stats
            
        if stats:
            acc_m = float(stats['classification_accuracy'][eval_pt]['mean'])
            acc_s = float(stats['classification_accuracy'][eval_pt]['std'])
            err_m = float(stats['target_error'][eval_pt]['mean'])
            err_s = float(stats['target_error'][eval_pt]['std'])
        else:
            acc_m, acc_s, err_m, err_s = 0.0, 0.0, 0.0, 0.0
            
        out['dynapse'][note] = {
            "accuracy_mean": acc_m, "accuracy_std": acc_s,
            "error_mean": err_m, "error_std": err_s
        }
        
    os.makedirs(os.path.dirname(filepath) or '.', exist_ok=True)
    with open(filepath, 'w') as f:
        json.dump(out, f, indent=4)



In [40]:
# Figure 3c: Yin Yang Bar Plots
csv_yinyang_sfc = '../simulation_results/yinyang/simulation_wandb_export_sfc.csv'
csv_yinyang_sg = '../simulation_results/yinyang/simulation_wandb_export_surrogategrad_MSE.csv'
csv_yinyang_bp = '../simulation_results/yinyang/simulation_wandb_export_backprop_crossentropy.csv'

if os.path.exists(csv_yinyang_sfc) and os.path.exists(csv_yinyang_sg):
    df_sfc = pd.read_csv(csv_yinyang_sfc, sep=';')
    df_bp = pd.read_csv(csv_yinyang_bp, sep=';')
    df_sg = pd.read_csv(csv_yinyang_sg, sep=';')
    if 'accuracy_classification' in df_sg.columns and 'test accuracy (Max)' not in df_sg.columns:
        df_sg.rename(columns={'accuracy_classification': 'test accuracy (Max)'}, inplace=True)
    df_merged = pd.concat([df_sfc, df_sg, df_bp], ignore_index=True)
    
    export_bar_plot_data(df_merged, yinyang_frozen_runs, yinyang_full_runs, 'summary_plots/fig3c_all.json')
    print("Exported fig3c_all.json successfully.")
else:
    print(f"Warning: Could not find one or more CSVs for YinYang bar plots.")



Exported fig3c_all.json successfully.


In [41]:
# Figure 2c: MNIST1D Bar Plots
csv_mnist1d_sfc = '../simulation_results/mnist1d/simulation_wandb_export_sfc.csv'
csv_mnist1d_sg = '../simulation_results/mnist1d/simulation_wandb_export_surrogategrad_crossentropy.csv'
csv_mnist1d_bp = '../simulation_results/mnist1d/simulation_wandb_export_backprop_crossentropy.csv'

if os.path.exists(csv_mnist1d_sfc) and os.path.exists(csv_mnist1d_sg):
    df_sfc_m = pd.read_csv(csv_mnist1d_sfc, sep=';')
    df_bp_m = pd.read_csv(csv_yinyang_bp, sep=';')
    df_sg_m = pd.read_csv(csv_mnist1d_sg, sep=';')
    if 'accuracy_classification' in df_sg_m.columns and 'test accuracy (Max)' not in df_sg_m.columns:
        df_sg_m.rename(columns={'accuracy_classification': 'test accuracy (Max)'}, inplace=True)
    df_merged_m = pd.concat([df_sfc_m, df_sg_m, df_bp_m], ignore_index=True)
    
    export_bar_plot_data(df_merged_m, mnist1d_frozen_runs, mnist1d_full_runs, 'summary_plots/fig2c_all.json')
    print("Exported fig2c_all.json successfully.")
else:
    print(f"Warning: Could not find one or more CSVs for MNIST1D bar plots.")



Exported fig2c_all.json successfully.


In [42]:
# Figure 2d: Weight Shift Distributions
import os
import pickle
import numpy as np
import json
from scipy.stats import gaussian_kde

def extract_weight_distributions(base_dir, filepath):
    weight_files = [
        ('Input_Hidden1', 'weights_in_hid1_fine.dat'),
        ('Hidden1_Hidden2', 'weights_hid1_hid2_fine.dat'),
        ('Hidden2_Output', 'weights_hid2_out_fine.dat')
    ]
    
    out = {}
    for layer_name, filename in weight_files:
        path = os.path.join(base_dir, filename)
        if not os.path.exists(path):
            print(f"File not found: {path}")
            continue
            
        # Extract name without extension
        name = os.path.splitext(os.path.basename(path))[0]
        
        # Load metadata
        with open(os.path.join(os.path.dirname(path), "trainlog.pkl"), "rb") as f:
            meta = pickle.load(f)
            
        shape = tuple(meta["arrays"][name])
        actual_shape = (-1,) + shape[1:] 
        
        # Load weights
        weights = np.fromfile(path, dtype="float64").reshape(actual_shape)
        weights_flat = weights.reshape(weights.shape[0], -1)
        
        w_start = weights_flat[0]
        w_final = weights_flat[-1]
        
        x_min = min(w_start.min(), w_final.min()) - 0.1
        x_max = max(w_start.max(), w_final.max()) + 0.1
        x_grid = np.linspace(x_min, x_max, 500)
        
        kde_start = gaussian_kde(w_start)
        y_start = kde_start(x_grid)
        
        kde_final = gaussian_kde(w_final)
        y_final = kde_final(x_grid)
        
        out[layer_name] = {
            "raw": {
                "initial": w_start.tolist(),
                "final": w_final.tolist()
            },
            "kde": {
                "x": x_grid.tolist(),
                "y_initial": y_start.tolist(),
                "y_final": y_final.tolist()
            }
        }
    
    os.makedirs(os.path.dirname(filepath) or '.', exist_ok=True)
    with open(filepath, 'w') as f:
        json.dump(out, f, indent=1)

# Usage
fig2d_base_dir = '../dynapse_results/mnist1d/full/201175_0'
if os.path.exists(fig2d_base_dir):
    extract_weight_distributions(fig2d_base_dir, 'summary_plots/fig2d_dynapse.json')
    print("Exported fig2d_dynapse.json successfully.")
else:
    print(f"Warning: Could not find directory {fig2d_base_dir} for weight distribution export.")



Exported fig2d_dynapse.json successfully.
